In [1]:
# 確認
import xgboost as xgb

print(xgb.__version__)

3.2.0


## ライブラリ・データ読み込み

In [ ]:
# ライブラリ
from pathlib import Path

import numpy as np
import pandas as pd

import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [3]:
# データ
ROOT = Path.cwd()

if ROOT.name == "smartphone_addiction":
    ROOT = ROOT.parent.parent

DATA_DIR = ROOT / "data" / "smartphone_addiction" / "raw"
OUTPUT_DIR = ROOT / "output" / "smartphone_addiction"

train = pd.read_csv(
    DATA_DIR / "train.csv"
)

test = pd.read_csv(
    DATA_DIR / "test.csv"
)

sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

TARGET = "addicted_label"

X = train.drop(
    columns=[TARGET, "id"]
).copy()

y = train[TARGET].copy()

X_test = test.drop(
    columns=["id"]
).copy()

In [4]:
# カテゴリ変数
categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(categorical_cols)

['gender', 'stress_level', 'academic_work_impact']


C:\Users\Owner\AppData\Local\Temp\ipykernel_20256\3095031101.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [5]:
# train/test でカテゴリーをそろえる
for col in categorical_cols:

    X[col] = X[col].fillna("Missing")
    X_test[col] = X_test[col].fillna("Missing")

    all_categories = pd.Index(
        pd.concat([
            X[col],
            X_test[col]
        ]).unique()
    )

    X[col] = pd.Categorical(
        X[col],
        categories=all_categories
    )

    X_test[col] = pd.Categorical(
        X_test[col],
        categories=all_categories
    )

In [6]:
# 確認
print(X.dtypes)

age                         float64
daily_screen_time_hours     float64
social_media_hours          float64
gaming_hours                float64
work_study_hours            float64
sleep_hours                 float64
notifications_per_day       float64
app_opens_per_day           float64
weekend_screen_time         float64
gender                     category
stress_level               category
academic_work_impact       category
dtype: object


## XGBoost baselineの設定

In [7]:
xgb_params = {
    "n_estimators": 5000,
    "learning_rate": 0.05,

    "max_depth": 6,
    "min_child_weight": 5,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.0,
    "reg_lambda": 1.0,

    "objective": "binary:logistic",
    "eval_metric": "auc",

    "tree_method": "hist",
    "enable_categorical": True,

    "random_state": 42,
    "n_jobs": -1,

    "early_stopping_rounds": 200
}

In [8]:
# 3-fold cv + OOF
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

oof_xgb = np.zeros(
    len(X)
)

test_pred_xgb = np.zeros(
    len(X_test)
)

fold_scores_xgb = []
best_iterations_xgb = []

In [9]:
# 実行
for fold, (
    train_idx,
    valid_idx
) in enumerate(
    cv.split(X, y),
    start=1
):

    print(
        f"\n===== Fold {fold} ====="
    )

    X_train = X.iloc[
        train_idx
    ]

    X_valid = X.iloc[
        valid_idx
    ]

    y_train = y.iloc[
        train_idx
    ]

    y_valid = y.iloc[
        valid_idx
    ]

    model = xgb.XGBClassifier(
        **xgb_params
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (
                X_valid,
                y_valid
            )
        ],

        verbose=500
    )

    valid_pred = (
        model.predict_proba(
            X_valid
        )[:, 1]
    )

    oof_xgb[
        valid_idx
    ] = valid_pred

    test_pred_xgb += (
        model.predict_proba(
            X_test
        )[:, 1]
        / cv.n_splits
    )

    fold_auc = roc_auc_score(
        y_valid,
        valid_pred
    )

    fold_scores_xgb.append(
        fold_auc
    )

    best_iterations_xgb.append(
        model.best_iteration
    )

    print(
        f"Fold {fold} AUC:"
        f" {fold_auc:.6f}"
    )

    print(
        "Best iteration:",
        model.best_iteration
    )


===== Fold 1 =====
[0]	validation_0-auc:0.92519
[500]	validation_0-auc:0.95894
[1000]	validation_0-auc:0.96228
[1500]	validation_0-auc:0.96308
[2000]	validation_0-auc:0.96335
[2500]	validation_0-auc:0.96342
[2739]	validation_0-auc:0.96341
Fold 1 AUC: 0.963428
Best iteration: 2539

===== Fold 2 =====
[0]	validation_0-auc:0.92581
[500]	validation_0-auc:0.95983
[1000]	validation_0-auc:0.96313
[1500]	validation_0-auc:0.96392
[2000]	validation_0-auc:0.96418
[2497]	validation_0-auc:0.96421
Fold 2 AUC: 0.964216
Best iteration: 2297

===== Fold 3 =====
[0]	validation_0-auc:0.92588
[500]	validation_0-auc:0.95980
[1000]	validation_0-auc:0.96303
[1500]	validation_0-auc:0.96388
[2000]	validation_0-auc:0.96415
[2309]	validation_0-auc:0.96418
Fold 3 AUC: 0.964185
Best iteration: 2109


## 結果確認

In [10]:
print(
    "Fold AUCs:",
    fold_scores_xgb
)

print(
    "Mean AUC:",
    np.mean(
        fold_scores_xgb
    )
)

print(
    "Std AUC:",
    np.std(
        fold_scores_xgb
    )
)

print(
    "Best iterations:",
    best_iterations_xgb
)

print(
    "OOF AUC:",
    roc_auc_score(
        y,
        oof_xgb
    )
)

Fold AUCs: [0.9634279599367781, 0.9642164297136318, 0.9641853134311685]
Mean AUC: 0.9639432343605261
Std AUC: 0.0003645754193444108
Best iterations: [2539, 2297, 2109]
OOF AUC: 0.9639383867788741


In [11]:
# 結果の保存
np.save(
    OUTPUT_DIR / "oof_xgboost.npy",
    oof_xgb
)

np.save(
    OUTPUT_DIR / "test_pred_xgboost.npy",
    test_pred_xgb
)

## モデルの予測相関を確認

In [12]:
oof_cb = np.load(
    OUTPUT_DIR
    / "oof_catboost_tuned.npy"
)

oof_lgb = np.load(
    OUTPUT_DIR
    / "oof_lightgbm.npy"
)

oof_xgb = np.load(
    OUTPUT_DIR
    / "oof_xgboost.npy"
)

In [13]:
prediction_df = pd.DataFrame({
    "CatBoost": oof_cb,
    "LightGBM": oof_lgb,
    "XGBoost": oof_xgb
})

display(
    prediction_df.corr()
)

,CatBoost,LightGBM,XGBoost
CatBoost,1.000000,0.992768,0.992774
LightGBM,0.992768,1.000000,0.994488
XGBoost,0.992774,0.994488,1.000000


## 3モデル均等平均

In [14]:
oof_blend_3 = (
    oof_cb
    + oof_lgb
    + oof_xgb
) / 3

In [15]:
auc_blend_3 = roc_auc_score(
    y,
    oof_blend_3
)

print(
    "CatBoost:",
    roc_auc_score(
        y,
        oof_cb
    )
)

print(
    "LightGBM:",
    roc_auc_score(
        y,
        oof_lgb
    )
)

print(
    "XGBoost:",
    roc_auc_score(
        y,
        oof_xgb
    )
)

print(
    "3-model ensemble:",
    auc_blend_3
)

CatBoost: 0.9636859867800479
LightGBM: 0.9636470878336719
XGBoost: 0.9639383867788741
3-model ensemble: 0.9644114338925592


## 少しだけ重みを変える

In [16]:
weight_candidates = [
    (0.4, 0.4, 0.2),
    (0.4, 0.3, 0.3),
    (0.3, 0.4, 0.3),
    (1/3, 1/3, 1/3)
]

ensemble_results = []

for cb_w, lgb_w, xgb_w in weight_candidates:

    pred = (
        cb_w * oof_cb
        + lgb_w * oof_lgb
        + xgb_w * oof_xgb
    )

    auc = roc_auc_score(
        y,
        pred
    )

    ensemble_results.append({
        "CatBoost_weight": cb_w,
        "LightGBM_weight": lgb_w,
        "XGBoost_weight": xgb_w,
        "AUC": auc
    })

ensemble_df = pd.DataFrame(
    ensemble_results
)

display(
    ensemble_df.sort_values(
        "AUC",
        ascending=False
    )
)

,CatBoost_weight,LightGBM_weight,XGBoost_weight,AUC
3,0.333333,0.333333,0.333333,0.964411
1,0.400000,0.300000,0.300000,0.964409
2,0.300000,0.400000,0.300000,0.964389
0,0.400000,0.400000,0.200000,0.964364


## submission

In [17]:
# 3モデルのtest予測を読み込み
test_pred_cb = np.load(
    OUTPUT_DIR / "test_pred_catboost_tuned.npy"
)

test_pred_lgb = np.load(
    OUTPUT_DIR / "test_pred_lightgbm.npy"
)

test_pred_xgb = np.load(
    OUTPUT_DIR / "test_pred_xgboost.npy"
)

In [18]:
# 均等平均
test_pred_ensemble = (
    test_pred_cb
    + test_pred_lgb
    + test_pred_xgb
) / 3

In [19]:
submission = sample_submission.copy()

submission[TARGET] = test_pred_ensemble

In [ ]:
# 保存
assert submission["id"].equals(test["id"])

assert submission[TARGET].notna().all()

assert submission[TARGET].between(
    0, 1
).all()

print(submission.shape)
display(submission.head())

(296302, 2)


,id,addicted_label
0,691369,0.999542
1,691370,0.950281
2,691371,0.954628
3,691372,0.990342
4,691373,0.997891


In [21]:
# 保存
submission_path = (
    OUTPUT_DIR
    / "submission_catboost_lightgbm_xgboost_ensemble.csv"
)

submission.to_csv(
    submission_path,
    index=False
)

print(submission_path)

c:\Users\Owner\Documents\projects\kaggle-study\output\smartphone_addiction\submission_catboost_lightgbm_xgboost_ensemble.csv
